# Local Alternative Bars — `plotly.local.alternative_bars`

This notebook shows how to use the `plotly.local.alternative_bars` style to visualise local alternative explanations from `calibrated-explanations`.

## Key interpretation rule

> **Each bar is an independent candidate explanation, not a component of a shared total.**
>
> Alternative 1 says: "if `age ≤ 42`, the model would predict X."  
> Alternative 2 says: "if `income > 50 000`, the model would predict Y."  
> These are separate hypothetical scenarios. Their bar values must not be summed.

The chart answers: *which alternative scenarios would move the prediction significantly, and in which direction?*

In [ ]:
import ce_visualization_plotly  # ensures plugin bootstrap runs
from calibrated_explanations import CalibratedExplainer
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

## 1 — Probabilistic classification example

In [ ]:
X, y = make_classification(
    n_samples=300, n_features=6, n_informative=4,
    n_redundant=0, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=42)
model.fit(X_train, y_train)

explainer = CalibratedExplainer(model, X_train, y_train, mode="classification", seed=42)

In [ ]:
# Generate alternative explanations for the first test instance
alternatives = explainer.explore_alternatives(X_test[:1])

# Default view: prediction delta per alternative, base prediction header at top
alternatives[0].plot(
    style="plotly.local.alternative_bars",
    show=True,
)

### Filtering and sorting

Show only the top 5 alternatives sorted by prediction movement (largest delta first):

In [ ]:
alternatives[0].plot(
    style="plotly.local.alternative_bars",
    filter_top=5,
    sort_by="prediction_delta",
    show=True,
)

### Role-based sort

Sort alternatives by role: counterfactuals first, then superfactuals, semifactuals, and unknowns:

In [ ]:
alternatives[0].plot(
    style="plotly.local.alternative_bars",
    sort_by="role",
    unknown_policy="hide",  # suppress alternatives with no assigned role
    show=True,
)

### HTML export

In [ ]:
alternatives[0].plot(
    style="plotly.local.alternative_bars",
    filter_top=10,
    sort_by="prediction_delta",
    show=False,
    path="alternative_bars_classification.html",
)

## 2 — Colour coding and role interpretation

| Colour | Role | Meaning |
|--------|------|---------|
| Blue | `counter` (counterfactual) | The alternative crosses the decision boundary |
| Green | `super` (superfactual) | The alternative reinforces the current prediction |
| Amber | `semi` (semifactual) | The alternative is on the same side but moves toward the boundary |
| Slate | `unknown` | Role information is unavailable or unresolved |

Component sub-bars (indented, lighter shade) appear for conjunctive rules that involve more than one feature.
All components of a conjunctive rule share the same prediction delta because per-feature decomposition is not available for alternative explanations — only the combined effect of the full rule is known.

## 3 — Suppressing conjunctive components

If you prefer one bar per alternative regardless of conjunction size:

In [ ]:
alternatives[0].plot(
    style="plotly.local.alternative_bars",
    include_conjunctive_components=False,
    show=True,
)

## 4 — Suppressing the prediction header

The base prediction sub-panel above the bars can be suppressed:

In [ ]:
alternatives[0].plot(
    style="plotly.local.alternative_bars",
    show_prediction_header=False,
    show=True,
)